In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import kagglehub
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# These two are separate libraries, not part of sklearn - need pip install if not already available
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load



# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md


# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [3]:
df = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')
df["TotalCharges"] = df["TotalCharges"].fillna(0)
pd.set_option("display.max_columns",None)

# print (df.head(10))
# print (df.info())

In [4]:
def feature_engineering(x_train , x_test):
    encoder = OneHotEncoder(sparse_output = False , handle_unknown ="ignore").set_output(transform="pandas")
    x_train_encoded = encoder.fit_transform(x_train)
    x_test_encoded = encoder.transform(x_test)
    return x_train_encoded , x_test_encoded

In [5]:
def feature_scaling(x_train,x_test):
    std = StandardScaler()
    x_train_scaled = pd.DataFrame(std.fit_transform(x_train),columns=x_train.columns,index=x_train.index)
    x_test_scaled = pd.DataFrame(std.transform(x_test),columns=x_test.columns,index=x_test.index)

    return x_train_scaled,x_test_scaled


In [6]:
yes_no_cols = [col for col in df.columns if set(df[col].dropna().unique()) == {'Yes', 'No'}]
print(yes_no_cols)

for col in yes_no_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']


In [7]:
def Classification(model,x_train,y_train,x_test,y_test):
    
    for name,model in model.items():
        model.fit(x_train,y_train)
        pred = model.predict(x_test)
        train_score = model.score(x_train,y_train)
    
        train_acc = model.score(x_train, y_train)
        test_acc = accuracy_score(y_test, pred)
        precision = precision_score(y_test, pred)
        recall = recall_score(y_test, pred)
        f1 = f1_score(y_test, pred)
        
        print(f"--- {name} ---")
        print(f"Train Accuracy : {train_acc:.4f}")
        print(f"Test Accuracy  : {test_acc:.4f}")
        print(f"Precision      : {precision:.4f}")
        print(f"Recall         : {recall:.4f}")
        print(f"F1 Score       : {f1:.4f}")
        print(f"Confusion Matrix:\n{confusion_matrix(y_test, pred)}\n")


In [8]:
   
x_train,x_test, y_train , y_test = train_test_split (df.drop(columns=["Churn","customerID"]),df["Churn"],random_state=42)

    
x_train_numeric = x_train.select_dtypes(include=['int64','float64'])
x_train_variable = x_train.select_dtypes(include=['object'])

x_test_numeric = x_test.select_dtypes(include=['int64','float64'])
x_test_variable = x_test.select_dtypes(include=['object'])

x_train_variable_encoded,x_test_variable_encoded = feature_engineering(x_train_variable,x_test_variable)

x_train_encoded = pd.concat([x_train_variable_encoded,x_train_numeric],axis=1)
x_test_encoded = pd.concat([x_test_variable_encoded,x_test_numeric],axis=1)

print(x_train_encoded.shape)
# print(df.corr(numeric_only=True)['Churn'].sort_values(ascending=False))

x_train_scaled,x_test_scaled = feature_scaling(x_train_encoded,x_test_encoded)

# Used to check columns with null values
# print(x_train_scaled.isnull().sum())


(5282, 41)


In [9]:
pipeline = Pipeline([
    ('model', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

param_grid = {
        'model__max_depth': [ 3, 4, 5, 6, 7, 8, 9, 10 , None],
        'model__min_samples_split': [2, 3, 4, 5,6,7,8,9,10],
        'model__min_samples_leaf': [1, 2, 4, 8],
        'model__max_features': ['sqrt', 'log2', None],
        'model__n_estimators': [100, 200]
            }

#Find Out parameters for max_depth and min_samples_split for Random Forest Regressor
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    )

grid_search.fit(x_train_scaled, y_train)
RDF_best_model = grid_search.best_estimator_.named_steps['model']
print("Best parameters found: ", grid_search.best_params_)


model = {
    "Logistic Regression":LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(**RDF_best_model.get_params()),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(random_state=42),
    "Neural Network (MLP)": MLPClassifier(random_state=42, max_iter=400)
}
Classification(model,x_train_scaled,y_train,x_test_scaled,y_test)

Best parameters found:  {'model__max_depth': 9, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 8, 'model__n_estimators': 200}
--- Logistic Regression ---
Train Accuracy : 0.7385
Test Accuracy  : 0.7553
Precision      : 0.5319
Recall         : 0.8351
F1 Score       : 0.6499
Confusion Matrix:
[[930 352]
 [ 79 400]]

--- KNN ---
Train Accuracy : 0.8359
Test Accuracy  : 0.7632
Precision      : 0.5738
Recall         : 0.5031
F1 Score       : 0.5362
Confusion Matrix:
[[1103  179]
 [ 238  241]]

--- Naive Bayes ---
Train Accuracy : 0.6865
Test Accuracy  : 0.7007
Precision      : 0.4727
Recall         : 0.8664
F1 Score       : 0.6116
Confusion Matrix:
[[819 463]
 [ 64 415]]

--- Decision Tree ---
Train Accuracy : 0.9985
Test Accuracy  : 0.7394
Precision      : 0.5215
Recall         : 0.5073
F1 Score       : 0.5143
Confusion Matrix:
[[1059  223]
 [ 236  243]]

--- Random Forest ---
Train Accuracy : 0.8254
Test Accuracy  : 0.7780
Precision      : 0.5659


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
